<a href="https://colab.research.google.com/github/AshokGit544/Airflow-Databricks-Finance-Control-Pipeline/blob/main/Airflow_Databricks_Finance_Control_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pandas numpy requests apache-airflow-providers-databricks==7.7.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.9/86.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.7/305.7 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.1/92.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.9/213.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.8/152.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [ ]:
import os
import random
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

PROJECT_DIR = Path("Enterprise-Finance-Control-Pipeline")
RAW_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "data" / "output"
DAG_DIR = PROJECT_DIR / "dags"
NOTEBOOK_DIR = PROJECT_DIR / "databricks_notebook"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DAG_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

print("Project folders created successfully")
print("Project path:", PROJECT_DIR.resolve())

Project folders created successfully
Project path: /content/Enterprise-Finance-Control-Pipeline


In [ ]:
vendors = pd.DataFrame([
    ["V001", "GridCore Services", "Maintenance", "MI"],
    ["V002", "PowerLine Analytics", "IT Services", "IL"],
    ["V003", "NorthPeak Electrical", "Electrical Components", "OH"],
    ["V004", "Prime Field Contractors", "Field Services", "IN"],
    ["V005", "Metro Utility Logistics", "Logistics", "WI"]
], columns=["vendor_id", "vendor_name", "vendor_category", "vendor_state"])

gl_accounts = pd.DataFrame([
    ["500100", "Maintenance Expense", "Expense"],
    ["500200", "IT Services Expense", "Expense"],
    ["500300", "Contractor Expense", "Expense"],
    ["500400", "Logistics Expense", "Expense"],
    ["210100", "Accounts Payable", "Liability"]
], columns=["gl_account", "gl_account_name", "gl_type"])

cost_centers = pd.DataFrame([
    ["CC100", "Transmission Ops", "Operations"],
    ["CC110", "Distribution Maintenance", "Operations"],
    ["CC120", "Substation Engineering", "Engineering"],
    ["CC130", "Finance Control", "Finance"],
    ["CC140", "Vendor Management", "Procurement"]
], columns=["cost_center_id", "cost_center_name", "department"])

descriptions = [
    "transformer maintenance work",
    "sap fico support service",
    "field repair contractor bill",
    "substation inspection work",
    "equipment logistics support",
    "invoice processing support",
    "network maintenance service"
]

invoice_rows = []
payment_rows = []
start_date = datetime(2025, 1, 1)

for i in range(1, 121):
    vendor = vendors.sample(1).iloc[0]
    gl = gl_accounts[gl_accounts["gl_type"] == "Expense"].sample(1).iloc[0]
    cc = cost_centers.sample(1).iloc[0]

    posting_date = start_date + timedelta(days=random.randint(0, 250))
    invoice_date = posting_date - timedelta(days=random.randint(1, 12))
    amount = round(random.uniform(1500, 60000), 2)

    invoice_id = f"INV{i:05d}"
    source_doc_id = f"SRC{i:05d}"

    invoice_rows.append([
        invoice_id,
        source_doc_id,
        vendor["vendor_id"],
        gl["gl_account"],
        cc["cost_center_id"],
        invoice_date.date().isoformat(),
        posting_date.date().isoformat(),
        amount,
        random.choice(descriptions)
    ])

    payment_rows.append([
        f"PAY{i:05d}",
        source_doc_id,
        invoice_id,
        (posting_date + timedelta(days=random.randint(5, 35))).date().isoformat(),
        round(amount * random.uniform(0.94, 1.00), 2)
    ])

invoices = pd.DataFrame(invoice_rows, columns=[
    "invoice_id", "source_doc_id", "vendor_id", "gl_account", "cost_center_id",
    "invoice_date", "posting_date", "amount_usd", "description"
])

payments = pd.DataFrame(payment_rows, columns=[
    "payment_id", "source_doc_id", "invoice_id", "payment_date", "paid_amount_usd"
])

invoices.loc[5, "vendor_id"] = None
invoices.loc[11, "amount_usd"] = -2500
invoices.loc[19, "description"] = None
invoices.loc[27, "cost_center_id"] = None

vendors.to_csv(RAW_DIR / "vendors.csv", index=False)
gl_accounts.to_csv(RAW_DIR / "gl_accounts.csv", index=False)
cost_centers.to_csv(RAW_DIR / "cost_centers.csv", index=False)
invoices.to_csv(RAW_DIR / "invoices.csv", index=False)
payments.to_csv(RAW_DIR / "payments.csv", index=False)

print("Sample source files created successfully")
print("vendors shape:", vendors.shape)
print("gl_accounts shape:", gl_accounts.shape)
print("cost_centers shape:", cost_centers.shape)
print("invoices shape:", invoices.shape)
print("payments shape:", payments.shape)

Sample source files created successfully
vendors shape: (5, 4)
gl_accounts shape: (5, 3)
cost_centers shape: (5, 3)
invoices shape: (120, 9)
payments shape: (120, 5)


In [ ]:
pyspark_notebook_code = r'''
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, concat_ws, when, trim, upper, round as spark_round

spark = SparkSession.builder.appName("EnterpriseFinanceControlPipeline").getOrCreate()

project_root = "Enterprise-Finance-Control-Pipeline"
base_path = f"{project_root}/data/raw"
output_path = f"{project_root}/data/output"

vendors = spark.read.option("header", True).csv(f"{base_path}/vendors.csv")
gl_accounts = spark.read.option("header", True).csv(f"{base_path}/gl_accounts.csv")
cost_centers = spark.read.option("header", True).csv(f"{base_path}/cost_centers.csv")
invoices = spark.read.option("header", True).csv(f"{base_path}/invoices.csv")
payments = spark.read.option("header", True).csv(f"{base_path}/payments.csv")

vendors = vendors.withColumn("vendor_name_clean", upper(trim(col("vendor_name"))))
invoices = invoices.withColumn("description_clean", upper(trim(col("description"))))

invoices = invoices.withColumn(
    "common_business_key",
    concat_ws("_", col("vendor_id"), col("cost_center_id"), col("gl_account"))
)

integrated = (
    invoices
    .join(vendors, on="vendor_id", how="left")
    .join(gl_accounts, on="gl_account", how="left")
    .join(cost_centers, on="cost_center_id", how="left")
    .join(
        payments.select("invoice_id", "payment_id", "payment_date", "paid_amount_usd"),
        on="invoice_id",
        how="left"
    )
)

dq_issues = integrated.select(
    "invoice_id",
    when(col("vendor_id").isNull(), lit("Missing vendor_id"))
    .when(col("cost_center_id").isNull(), lit("Missing cost_center_id"))
    .when(col("amount_usd").cast("double") <= 0, lit("Invalid amount"))
    .when(col("description").isNull(), lit("Missing description"))
    .otherwise(lit(None)).alias("issue_reason")
).filter(col("issue_reason").isNotNull())

exception_summary = integrated.groupBy("cost_center_id").count()

payment_reconciliation = integrated.withColumn(
    "amount_usd_double", col("amount_usd").cast("double")
).withColumn(
    "paid_amount_usd_double", col("paid_amount_usd").cast("double")
).withColumn(
    "payment_gap", spark_round(col("amount_usd_double") - col("paid_amount_usd_double"), 2)
).select(
    "invoice_id",
    "vendor_id",
    "cost_center_id",
    "gl_account",
    "amount_usd_double",
    "paid_amount_usd_double",
    "payment_gap"
)

retrieval_ready = integrated.withColumn(
    "retrieval_text",
    concat_ws(
        " ",
        lit("Invoice"),
        col("invoice_id"),
        lit("Vendor"),
        col("vendor_name"),
        lit("Department"),
        col("department"),
        lit("Cost Center"),
        col("cost_center_name"),
        lit("GL Account"),
        col("gl_account_name"),
        lit("Invoice Amount"),
        col("amount_usd"),
        lit("Paid Amount"),
        col("paid_amount_usd"),
        lit("Description"),
        col("description")
    )
).select("invoice_id", "common_business_key", "retrieval_text")

integrated.toPandas().to_csv(f"{output_path}/integrated_finance_view.csv", index=False)
dq_issues.toPandas().to_csv(f"{output_path}/data_quality_issues.csv", index=False)
exception_summary.toPandas().to_csv(f"{output_path}/exception_summary.csv", index=False)
payment_reconciliation.toPandas().to_csv(f"{output_path}/payment_reconciliation.csv", index=False)
retrieval_ready.toPandas().to_csv(f"{output_path}/retrieval_ready_output.csv", index=False)

print("Databricks-style PySpark pipeline completed successfully")
spark.stop()
'''
(NOTEBOOK_DIR / "finance_control_databricks_notebook.py").write_text(pyspark_notebook_code, encoding="utf-8")

print("Databricks-style notebook file created successfully")
print((NOTEBOOK_DIR / "finance_control_databricks_notebook.py").resolve())

Databricks-style notebook file created successfully
/content/Enterprise-Finance-Control-Pipeline/databricks_notebook/finance_control_databricks_notebook.py


In [ ]:
!pip -q install pyspark

In [ ]:
!python Enterprise-Finance-Control-Pipeline/databricks_notebook/finance_control_databricks_notebook.py

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/31 21:58:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Databricks-style PySpark pipeline completed successfully


In [ ]:
import pandas as pd

output_path = "Enterprise-Finance-Control-Pipeline/data/output"

files = [
    "integrated_finance_view.csv",
    "data_quality_issues.csv",
    "exception_summary.csv",
    "payment_reconciliation.csv",
    "retrieval_ready_output.csv"
]

for file in files:
    print("\n")
    print(f"Preview of {file}")
    print("")
    df = pd.read_csv(f"{output_path}/{file}")
    print(df.head(5))



Preview of integrated_finance_view.csv

  invoice_id cost_center_id  gl_account vendor_id source_doc_id invoice_date  \
0   INV00001          CC140      500200      V002      SRC00001   2025-06-11   
1   INV00002          CC100      500100      V005      SRC00002   2025-07-06   
2   INV00003          CC110      500100      V005      SRC00003   2025-01-08   
3   INV00004          CC100      500100      V001      SRC00004   2025-05-20   
4   INV00005          CC120      500200      V002      SRC00005   2025-04-15   

  posting_date  amount_usd                   description  \
0   2025-06-13     2963.13  field repair contractor bill   
1   2025-07-08    41086.92   equipment logistics support   
2   2025-01-09     6981.17      sap fico support service   
3   2025-05-24    43387.15    invoice processing support   
4   2025-04-25    17774.16   network maintenance service   

              description_clean  ... vendor_category vendor_state  \
0  FIELD REPAIR CONTRACTOR BILL  ...     IT Ser

In [ ]:
airflow_databricks_dag_code = r'''
from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.databricks.operators.databricks import (
    DatabricksCreateJobsOperator,
    DatabricksRunNowOperator,
)
from datetime import datetime

default_args = {
    "owner": "Ashok",
    "start_date": datetime(2025, 1, 1),
    "retries": 1
}

job_config = {
    "name": "enterprise_finance_control_databricks_job",
    "tasks": [
        {
            "task_key": "run_finance_control_notebook",
            "notebook_task": {
                "notebook_path": "/Workspace/Users/your_email@company.com/finance_control_databricks_notebook"
            },
            "new_cluster": {
                "spark_version": "15.4.x-scala2.12",
                "node_type_id": "Standard_DS3_v2",
                "num_workers": 1
            }
        }
    ]
}

with DAG(
    dag_id="enterprise_finance_control_airflow_to_databricks",
    default_args=default_args,
    schedule="@daily",
    catchup=False,
    tags=["airflow", "databricks", "finance", "control"]
) as dag:

    start_task = EmptyOperator(task_id="start_pipeline")

    create_or_update_job = DatabricksCreateJobsOperator(
        task_id="create_or_update_databricks_job",
        databricks_conn_id="databricks_default",
        json=job_config
    )

    trigger_databricks_job = DatabricksRunNowOperator(
        task_id="trigger_databricks_job",
        databricks_conn_id="databricks_default",
        job_id="{{ ti.xcom_pull(task_ids='create_or_update_databricks_job')['job_id'] }}"
    )

    end_task = EmptyOperator(task_id="end_pipeline")

    start_task >> create_or_update_job >> trigger_databricks_job >> end_task
'''

dag_file_path = DAG_DIR / "enterprise_finance_control_airflow_to_databricks.py"
dag_file_path.write_text(airflow_databricks_dag_code, encoding="utf-8")

print("Airflow DAG file created successfully")
print("DAG path:", dag_file_path.resolve())

Airflow DAG file created successfully
DAG path: /content/Enterprise-Finance-Control-Pipeline/dags/enterprise_finance_control_airflow_to_databricks.py


In [ ]:
print(dag_file_path.read_text(encoding="utf-8"))


from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.databricks.operators.databricks import (
    DatabricksCreateJobsOperator,
    DatabricksRunNowOperator,
)
from datetime import datetime

default_args = {
    "owner": "Ashok",
    "start_date": datetime(2025, 1, 1),
    "retries": 1
}

job_config = {
    "name": "enterprise_finance_control_databricks_job",
    "tasks": [
        {
            "task_key": "run_finance_control_notebook",
            "notebook_task": {
                "notebook_path": "/Workspace/Users/your_email@company.com/finance_control_databricks_notebook"
            },
            "new_cluster": {
                "spark_version": "15.4.x-scala2.12",
                "node_type_id": "Standard_DS3_v2",
                "num_workers": 1
            }
        }
    ]
}

with DAG(
    dag_id="enterprise_finance_control_airflow_to_databricks",
    default_args=default_args,
    schedule="@daily",
    catchu

In [7]:
#Personal folder files validation
import zipfile

zip_path = Path("Enterprise-Finance-Control-Pipeline-final.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in PROJECT_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(PROJECT_DIR))

print("Project zipped successfully")
print("ZIP path:", zip_path.resolve())

Project zipped successfully
ZIP path: /content/Enterprise-Finance-Control-Pipeline-final.zip


In [4]:
print(os.listdir("/content"))

['.config', 'Enterprise-Finance-Control-Pipeline', 'sample_data']


In [5]:
for p in PROJECT_DIR.rglob("*"):
    print(p)

Enterprise-Finance-Control-Pipeline/dags
Enterprise-Finance-Control-Pipeline/data
Enterprise-Finance-Control-Pipeline/databricks_notebook
Enterprise-Finance-Control-Pipeline/data/output
Enterprise-Finance-Control-Pipeline/data/raw
